# Notebook 13 — nnU-Net INCA Baseline

Single-run nnU-Net reference on INCA dataset (634 train frames).
Contextual comparison with the U-Net++ SSL pipeline.


In [ ]:
# === Cell 1: Mount Drive, install nnU-Net v2, set env vars ===

from google.colab import drive
drive.mount("/content/drive")

!pip install nnunetv2

import os

WORKSPACE = "/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace"
os.makedirs(os.path.join(WORKSPACE, "nnUNet_raw"), exist_ok=True)
os.makedirs(os.path.join(WORKSPACE, "nnUNet_preprocessed"), exist_ok=True)
os.makedirs(os.path.join(WORKSPACE, "nnUNet_results_inca"), exist_ok=True)

os.environ["nnUNet_raw"] = os.path.join(WORKSPACE, "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = os.path.join(WORKSPACE, "nnUNet_preprocessed")
os.environ["nnUNet_results"] = os.path.join(WORKSPACE, "nnUNet_results_inca")

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])


In [ ]:
# === Cell 2: Convert INCA dataset to nnU-Net format (Dataset504_VFSS_INCA) ===

import os, shutil, json
from PIL import Image
import numpy as np

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
RAW = os.environ["nnUNet_raw"]
DATASET = os.path.join(RAW, "Dataset504_VFSS_INCA")

imagesTr = os.path.join(DATASET, "imagesTr")
labelsTr = os.path.join(DATASET, "labelsTr")
imagesTs = os.path.join(DATASET, "imagesTs")

for d in [imagesTr, labelsTr, imagesTs]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

# Copy train + val images; remap masks to {0,1}
num_training = 0
for split in ["train", "val"]:
    img_dir = os.path.join(BASE, split, "images")
    msk_dir = os.path.join(BASE, split, "masks")
    for fname in sorted(os.listdir(img_dir)):
        if not fname.endswith(".png"):
            continue
        stem = fname.replace(".png", "")
        # nnU-Net expects: {stem}_0000.png for images, {stem}.png for labels
        img = Image.open(os.path.join(img_dir, fname)).convert("L")
        img.save(os.path.join(imagesTr, f"{stem}_0000.png"))
        msk = np.array(Image.open(os.path.join(msk_dir, fname)).convert("L"))
        msk_bin = (msk > 127).astype(np.uint8)  # remap to {0, 1}
        Image.fromarray(msk_bin).save(os.path.join(labelsTr, f"{stem}.png"))
        num_training += 1

# Copy test images
test_img_dir = os.path.join(BASE, "test", "images")
for fname in sorted(os.listdir(test_img_dir)):
    if not fname.endswith(".png"):
        continue
    stem = fname.replace(".png", "")
    img = Image.open(os.path.join(test_img_dir, fname)).convert("L")
    img.save(os.path.join(imagesTs, f"{stem}_0000.png"))

# dataset.json
ds_json = {
    "channel_names": {"0": "Xray"},
    "labels": {"background": 0, "vertebra": 1},
    "numTraining": num_training,
    "file_ending": ".png",
}
with open(os.path.join(DATASET, "dataset.json"), "w") as f:
    json.dump(ds_json, f, indent=2)

print(f"Dataset504_VFSS_INCA: {num_training} training, "
      f"{len([f for f in os.listdir(imagesTs) if f.endswith('.png')])} test")


In [ ]:
# === Cell 3: Plan, preprocess, write custom splits ===

!nnUNetv2_plan_and_preprocess -d 504 --verify_dataset_integrity -c 2d

import os, json

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"

# Build train/val split matching our pipeline
train_stems = sorted([
    f.replace(".png", "")
    for f in os.listdir(os.path.join(BASE, "train", "images"))
    if f.endswith(".png")
])
val_stems = sorted([
    f.replace(".png", "")
    for f in os.listdir(os.path.join(BASE, "val", "images"))
    if f.endswith(".png")
])

print(f"Train: {len(train_stems)}, Val: {len(val_stems)}")
assert not set(train_stems) & set(val_stems), "Train/val overlap!"

splits_path = os.path.join(
    os.environ["nnUNet_preprocessed"],
    "Dataset504_VFSS_INCA",
    "splits_final.json"
)
with open(splits_path, "w") as f:
    json.dump([{"train": train_stems, "val": val_stems}], f, indent=2)
print(f"splits_final.json written: {splits_path}")


In [ ]:
# === Cell 4: Train nnU-Net on INCA ===

import os, time, torch, gc
gc.collect()
torch.cuda.empty_cache()

WORKSPACE = "/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace"
results_dir = os.path.join(WORKSPACE, "nnUNet_results_inca")
os.makedirs(results_dir, exist_ok=True)
os.environ["nnUNet_results"] = results_dir
print(f"nnUNet_results -> {results_dir}")

t0 = time.time()
!nnUNetv2_train 504 2d 0
# To resume if Colab disconnects: !nnUNetv2_train 504 2d 0 --c
elapsed = time.time() - t0

fold_dir = os.path.join(
    results_dir,
    "Dataset504_VFSS_INCA", "nnUNetTrainer__nnUNetPlans__2d", "fold_0"
)
print(f"\nDone in {elapsed/3600:.1f}h")
print(f"Checkpoints: {fold_dir}")
if os.path.isdir(fold_dir):
    print("Contents:", os.listdir(fold_dir))


In [ ]:
# === Cell 5: Predict + Evaluate ===

import os, json, csv, re, shutil
import numpy as np
from PIL import Image

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
WORKSPACE = os.path.join(BASE, "nnunet_workspace")
RESULTS_BASE = os.path.join(BASE, "nnunet_baseline_results")
os.makedirs(RESULTS_BASE, exist_ok=True)

os.environ["nnUNet_results"] = os.path.join(WORKSPACE, "nnUNet_results_inca")

# Predict
RAW = os.environ["nnUNet_raw"]
imagesTs = os.path.join(RAW, "Dataset504_VFSS_INCA", "imagesTs")
pred_dir = os.path.join(RESULTS_BASE, "predictions_inca")
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict \
    -i {imagesTs} \
    -o {pred_dir} \
    -d 504 -c 2d -f 0

# Evaluate
GT_DIR = os.path.join(BASE, "data", "inca_dataset", "test", "masks")
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".png")])
gt_by_stem = {f.replace(".png", ""): os.path.join(GT_DIR, f) for f in gt_files}

def normalize_pred_stem(fname):
    stem = fname.replace(".png", "")
    stem = re.sub(r"_\d{4}$", "", stem)
    return stem

def compute_dice_iou(pred_bin, gt_bin):
    intersection = np.sum(pred_bin * gt_bin)
    sum_pred = np.sum(pred_bin)
    sum_gt = np.sum(gt_bin)
    dice = (2.0 * intersection) / (sum_pred + sum_gt) if (sum_pred + sum_gt) > 0 else 1.0
    union = sum_pred + sum_gt - intersection
    iou = intersection / union if union > 0 else 1.0
    return dice, iou

pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith(".png")])
pred_by_stem = {normalize_pred_stem(f): os.path.join(pred_dir, f) for f in pred_files}
matched = sorted(set(pred_by_stem.keys()) & set(gt_by_stem.keys()))
print(f"Matched: {len(matched)} of {len(gt_files)} test images")

per_image = []
for stem in matched:
    gt = np.array(Image.open(gt_by_stem[stem]).convert("L"))
    pred = np.array(Image.open(pred_by_stem[stem]).convert("L"))
    if pred.shape != gt.shape:
        pred = np.array(Image.fromarray(pred).resize((gt.shape[1], gt.shape[0]), Image.NEAREST))
    gt_bin = (gt > 0).astype(np.float32)
    pred_bin = (pred > 0).astype(np.float32)
    dice, iou = compute_dice_iou(pred_bin, gt_bin)
    per_image.append({"stem": stem, "f1": dice, "iou": iou})

mean_f1 = float(np.mean([r["f1"] for r in per_image]))
mean_iou = float(np.mean([r["iou"] for r in per_image]))

# Save
csv_path = os.path.join(RESULTS_BASE, "per_image_metrics_inca.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["stem", "f1", "iou"])
    writer.writeheader()
    writer.writerows(per_image)

metrics = {
    "condition": "nnunet_inca_100pct",
    "real_train": 634 + 140,
    "mean_f1": mean_f1,
    "mean_iou": mean_iou,
    "num_test_images": len(per_image),
}
with open(os.path.join(RESULTS_BASE, "metrics_inca.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nnnU-Net INCA 100%: F1={mean_f1:.4f}, IoU={mean_iou:.4f} ({len(per_image)} images)")
print(f"Saved: {csv_path}")
